# Walmart Sales Forecasting

This notebook performs sales forecasting analysis using the pre-trained AutoGluon TimeSeries model from the pipeline, and visualizes predictions with 95%, 85%, and 75% confidence intervals using Plotly.

In [1]:
import os
import yaml
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

## Load Configuration

We load settings from the pipeline configuration file to avoid hardcoding variables.

In [2]:
with open('../config/walmart_sales.yaml', 'r') as f:
    config = yaml.safe_load(f)

pipeline_name = config["pipeline_name"]
target_column = config["model"]["target_column"]
eval_metric = config["model"]["eval_metric"]
prediction_length = config["model"]["prediction_length"]

path = config["data"]["processed_path"]
if path.startswith("/data/") and not os.path.exists("/data"):
    if os.path.basename(os.getcwd()) == "notebooks":
        processed_path = os.path.abspath(os.path.join("..", path.lstrip("/")))
    else:
        processed_path = os.path.abspath(path.lstrip("/"))
else:
    processed_path = os.path.abspath(path)

model_path = f"../models/{pipeline_name}"

## Data Loading

We load the processed Walmart sales dataset and convert it to the format required by AutoGluon (`TimeSeriesDataFrame`).

In [3]:
df = pd.read_csv(processed_path)
train_data = TimeSeriesDataFrame.from_data_frame(
    df, id_column="store", timestamp_column="date"
)

## Dataset Splitting

We split the data into training and test sets using the same prediction length configured in the pipeline.

In [4]:
train_split, test_split = train_data.train_test_split(prediction_length)

## Load Pre-trained Model

We load the pre-trained TimeSeriesPredictor model from the pipeline output directory.

In [5]:
predictor = TimeSeriesPredictor.load(model_path)

## Predictions & Confidence Intervals Estimation

We generate predictions on the training split and estimate the 95%, 85%, and 75% confidence intervals using the predicted quantiles (0.1 and 0.9) assuming a normal distribution.

In [6]:
predictions = predictor.predict(train_split)
sigma = (predictions['0.9'] - predictions['0.1']) / (2 * 1.28155)
predictions['0.975'] = predictions['mean'] + 1.95996 * sigma
predictions['0.025'] = predictions['mean'] - 1.95996 * sigma
predictions['0.925'] = predictions['mean'] + 1.43953 * sigma
predictions['0.075'] = predictions['mean'] - 1.43953 * sigma
predictions['0.875'] = predictions['mean'] + 1.15035 * sigma
predictions['0.125'] = predictions['mean'] - 1.15035 * sigma

## Evaluation of Main Metrics

We evaluate the model on the test split and show the leaderboard.

In [7]:
evaluation_results = predictor.evaluate(test_split)
leaderboard = predictor.leaderboard(test_split)
print("Evaluation Metrics:")
for k, v in evaluation_results.items():
    print(f"{k}: {abs(v):.5f}")

Evaluation Metrics:
WAPE: 0.03470


In [8]:
leaderboard

,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,Chronos2,-0.028662,-0.034291,1.087742,0.395720,2.376024,9
1,WeightedEnsemble,-0.034699,-0.029678,1.261468,0.539156,0.320948,19
2,TiDE,-0.040585,-0.032995,0.042651,0.032895,75.007529,13
3,RecursiveTabular_3,-0.042459,-0.042505,0.052798,0.067343,0.789124,5
4,PatchTST,-0.043774,-0.032278,0.029949,0.019470,34.250613,12
5,WaveNet,-0.044025,-0.050648,0.188206,0.081232,13.632339,14
6,ADIDA,-0.046620,-0.059797,0.894091,2.236326,0.009560,15
7,RecursiveTabular,-0.046624,-0.044680,0.197501,0.045957,0.299998,3
8,RecursiveTabular_2,-0.046690,-0.046357,0.070091,0.047614,0.438695,4
9,TemporalFusionTransformer,-0.047072,-0.046474,0.046930,0.022089,27.242727,10


## Prediction Interval Coverage Analysis

We analyze the coverage of the predicted confidence intervals (95%, 85%, and 75%) against the actual test split sales.

In [9]:
y_true = test_split.slice_by_timestep(-prediction_length, None)[target_column]
within_95 = (predictions['0.025'] <= y_true) & (y_true <= predictions['0.975'])
within_85 = (predictions['0.075'] <= y_true) & (y_true <= predictions['0.925'])
within_75 = (predictions['0.125'] <= y_true) & (y_true <= predictions['0.875'])

stores_100_95 = within_95.groupby('item_id').all().mean()
stores_100_85 = within_85.groupby('item_id').all().mean()
stores_100_75 = within_75.groupby('item_id').all().mean()

summary_df = pd.DataFrame({
    'Confidence Interval': ['95%', '85%', '75%'],
    'Expected Coverage': [0.95, 0.85, 0.75],
    'Actual Counts': [within_95.sum(), within_85.sum(), within_75.sum()],
    'Actual Percentage': [within_95.mean(), within_85.mean(), within_75.mean()],
    'Actual Error': [1.0 - within_95.mean(), 1.0 - within_85.mean(), 1.0 - within_75.mean()],
    'Stores 100% Correct': [stores_100_95, stores_100_85, stores_100_75]
})

In [10]:
summary_df

,Confidence Interval,Expected Coverage,Actual Counts,Actual Percentage,Actual Error,Stores 100% Correct
0,95%,0.95,570,0.974359,0.025641,0.755556
1,85%,0.85,538,0.919658,0.080342,0.488889
2,75%,0.75,492,0.841026,0.158974,0.244444


## Store-level Performance Visualization & PDF Export

We calculate WAPE for each store, plot two 3x3 grids for the 9 worst and 9 best stores, and export them as PDF files.

In [11]:
y_pred = predictions['mean']
abs_error = (y_true - y_pred).abs()
store_wape = abs_error.groupby('item_id').sum() / y_true.groupby('item_id').sum()
store_wape = store_wape.sort_values()

def plot_store_grid(store_ids, title, filename):
    fig = make_subplots(
        rows=5, cols=2,
        subplot_titles=[f"Store {sid} (WAPE: {store_wape[sid]:.3f})" for sid in store_ids],
        vertical_spacing=0.05,
        horizontal_spacing=0.08
    )
    
    start_date = None
    end_date = None
    
    for idx, store_id in enumerate(store_ids):
        row = idx // 2 + 1
        col = idx % 2 + 1
        
        history = df[df['store'] == store_id].copy()
        history['date'] = pd.to_datetime(history['date'])
        history = history.sort_values('date')
        
        pred_store = predictions.loc[store_id]
        pred_dates = pd.to_datetime(pred_store.index.get_level_values('timestamp'))
        
        history_train = history[history['date'] < pred_dates[0]]
        start_date = history_train['date'].iloc[-6].strftime('%Y-%m-%d')
        end_date = pred_dates[-1].strftime('%Y-%m-%d')
        
        fig.add_trace(go.Scatter(
            x=history['date'],
            y=history[target_column],
            name='Actual Sales',
            mode='lines',
            line=dict(color='#1f77b4', width=1.5),
            showlegend=(idx == 0)
        ), row=row, col=col)
        
        fig.add_trace(go.Scatter(
            x=pred_dates,
            y=pred_store['mean'],
            name='Predicted Sales (Mean)',
            mode='lines+markers',
            line=dict(color='#ff7f0e', width=1.5, dash='dash'),
            marker=dict(size=4),
            showlegend=(idx == 0)
        ), row=row, col=col)
        
        fig.add_trace(go.Scatter(
            x=np.concatenate([pred_dates, pred_dates[::-1]]),
            y=np.concatenate([pred_store['0.975'], pred_store['0.025'][::-1]]),
            fill='toself',
            fillcolor='rgba(255, 127, 14, 0.1)',
            line=dict(color='rgba(255, 127, 14, 0)'),
            hoverinfo="skip",
            showlegend=(idx == 0),
            name='95% Confidence Interval'
        ), row=row, col=col)
        
        fig.add_trace(go.Scatter(
            x=np.concatenate([pred_dates, pred_dates[::-1]]),
            y=np.concatenate([pred_store['0.925'], pred_store['0.075'][::-1]]),
            fill='toself',
            fillcolor='rgba(255, 127, 14, 0.15)',
            line=dict(color='rgba(255, 127, 14, 0)'),
            hoverinfo="skip",
            showlegend=(idx == 0),
            name='85% Confidence Interval'
        ), row=row, col=col)
        
        fig.add_trace(go.Scatter(
            x=np.concatenate([pred_dates, pred_dates[::-1]]),
            y=np.concatenate([pred_store['0.875'], pred_store['0.125'][::-1]]),
            fill='toself',
            fillcolor='rgba(255, 127, 14, 0.2)',
            line=dict(color='rgba(255, 127, 14, 0)'),
            hoverinfo="skip",
            showlegend=(idx == 0),
            name='75% Confidence Interval'
        ), row=row, col=col)
        
    fig.update_layout(
        title=dict(text=title, font=dict(size=20)),
        height=1500,
        template="plotly_dark",
        margin=dict(l=40, r=40, t=100, b=40)
    )
    fig.show()
    fig.update_xaxes(range=[start_date, end_date])
    fig.write_image(filename)

worst_stores = store_wape.tail(10).index.tolist()[::-1]
best_stores = store_wape.head(10).index.tolist()

plot_store_grid(worst_stores, "Top 10 Stores with Worst Forecasting Results (Highest WAPE)", "../figures/worst_stores_forecast.pdf")
plot_store_grid(best_stores, "Top 10 Stores with Best Forecasting Results (Lowest WAPE)", "../figures/best_stores_forecast.pdf")
